[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_pocket_detection.ipynb)

# Finding binding pockets with P2Rank

**Orange group · Tuberculosis**

A drug can only work on a protein if it has somewhere to sit: a **pocket**, a cleft in
the surface where a small molecule fits and stays. This notebook uses P2Rank to find
the pockets in every target's structure and to score how likely each one is to bind a
drug-like molecule, which is a first, computational look at druggability.

## What you will do

- Download the structures chosen in the previous notebook.
- Install P2Rank, a machine-learning tool that predicts binding pockets.
- Check P2Rank on InhA, a target whose drug-binding site is known.
- Predict pockets for every target and summarise them per protein.
- Rank the targets by their best pocket and save the result.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the structures

The previous notebook, `orange_protein_structures`, chose one structure per target and
saved where each came from. That table has been copied into `data/`. The structure
files themselves are not stored anywhere, so we download them again.

First the packages, and the probability above which we call a pocket *likely*.

In [ ]:
import os

import pandas as pd
import stylia
from scripts import pockets, structures

# Plots: slide format, Ersilia colours
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

# A pocket with at least this probability is considered likely to bind a ligand
MIN_PROBABILITY = 0.5
print(f"likely pocket: probability of at least {MIN_PROBABILITY}")

Read the table. Each target uses either a PDB structure or an AlphaFold model.

In [ ]:
table = pd.read_csv("data/mtb_targets_structures.csv")
print(f"{len(table)} targets")
table["structure_source"].value_counts()

Download the structure files. The same function as in the previous notebook is used, so the files are identical. It takes a couple of minutes, or a few seconds if they are already here.

In [ ]:
table["structure_file"] = structures.download_all(table)
missing = ~table["structure_file"].map(os.path.exists)
print(f"{len(table) - missing.sum()} of {len(table)} structure files ready")

## 2. Set up P2Rank

[P2Rank](https://github.com/rdk/p2rank) (Krivák and Hoksza, 2018) predicts where small
molecules bind on a protein, using only its structure. It works in three steps:

1. It covers the surface of the protein with evenly spaced points.
2. For each point it describes the neighbourhood: which atoms are nearby, their
   chemistry, and how buried the point is.
3. A **random forest**, a machine-learning model trained on thousands of real
   protein-ligand complexes, gives each point a score. Neighbouring high-scoring points
   are grouped into pockets.

It does not need to know anything about the protein in advance, so it can also find
pockets away from the active site. Those are candidate **allosteric** sites, where a
molecule can change a protein's activity without competing for its active site.

P2Rank is written in Java, not Python. `install_p2rank` downloads it (about 275 MB,
once per session) and `find_java` finds Java 17 or newer, installing it first on
Colab if needed. Together they take about a minute.

In [ ]:
prank = pockets.install_p2rank()
print(f"P2Rank: {prank}")
print(f"Java:   {pockets.find_java()}")

## 3. Try it on one protein

Before trusting a tool on 348 proteins, we try it on one where we already know the
answer. InhA binds its cofactor NADH and the activated form of isoniazid in one
pocket. Four residues lining that pocket are phenylalanine 149, tyrosine 158, lysine
165 and methionine 199.

In [ ]:
inha = table[table["gene_name"] == "inhA"].iloc[0]
pockets.run_p2rank(prank, [inha["structure_file"]], "work/p2rank_inha")
inha_pockets = pockets.read_predictions("work/p2rank_inha")
inha_pockets[["rank", "score", "probability", "sas_points", "residue_ids"]]

Each row is one pocket, best first:

- **`score`** is P2Rank's raw score. Higher is better, but it has no fixed scale.
- **`probability`** turns the score into a number between 0 and 1: roughly, how likely
  the pocket is to be a real binding site. This is the column we will use.
- **`sas_points`** is the number of surface points in the pocket, a rough measure of
  its size.
- **`residue_ids`** lists the residues lining the pocket, as `chain_number`.

Is the known binding site in the top pocket?

In [ ]:
KNOWN_SITE = {"A_149": "Phe149", "A_158": "Tyr158", "A_165": "Lys165", "A_199": "Met199"}
top_residues = inha_pockets.iloc[0]["residue_ids"].split()
{name: residue in top_residues for residue, name in KNOWN_SITE.items()}

All four are in the top pocket, and P2Rank gives it a high probability. Without being
told anything about InhA, it found the pocket where isoniazid works.

## 4. Run it on every target

P2Rank comes with more than one trained model, called **profiles**:

- The **default** profile was trained on X-ray crystal structures. One of the things it
  uses is the B-factor, which in an X-ray structure says how much each atom wobbles.
- The **alphafold** profile does not use the B-factor. In AlphaFold models that column
  holds the pLDDT confidence instead, and in cryo-EM and NMR structures it means
  something different again. The P2Rank authors recommend this profile for all three.

So each target gets the profile that suits its structure.

In [ ]:
is_xray = table["method"].eq("X-ray diffraction") & table["structure_source"].eq("PDB")
table["p2rank_profile"] = is_xray.map({True: "default", False: "alphafold"})
table["p2rank_profile"].value_counts()

Run P2Rank twice, once per profile. Each run reads all of its structures in one go, so the whole shortlist takes one or two minutes.

In [ ]:
pockets.run_p2rank(prank, table.loc[is_xray, "structure_file"], "work/p2rank/default")
pockets.run_p2rank(prank, table.loc[~is_xray, "structure_file"], "work/p2rank/alphafold",
                   config="alphafold")
all_pockets = pd.concat([pockets.read_predictions(f"work/p2rank/{profile}")
                         for profile in ["default", "alphafold"]], ignore_index=True)
print(f"{len(all_pockets):,} pockets in {all_pockets['uniprot_ac'].nunique()} proteins")

## 5. Read the results

We now have one row per pocket. To compare targets we need one row per protein:
how many pockets it has, how many are likely (probability at least
`MIN_PROBABILITY`), and how good the best one is.

In [ ]:
summary = pockets.summarise(all_pockets, MIN_PROBABILITY)
ranked = table.merge(summary, on="uniprot_ac", how="left")
counts = ["n_pockets", "n_likely_pockets"]
ranked[counts] = ranked[counts].fillna(0).astype(int)
ranked[["gene_name", "structure_source", "n_pockets", "n_likely_pockets",
        "top_probability"]].head()

Some proteins have no pocket at all. Which ones?

In [ ]:
no_pocket = ranked[ranked["n_pockets"] == 0]
print(f"{len(no_pocket)} of {len(ranked)} proteins have no pocket")
no_pocket[["gene_name", "protein_name", "structure_source"]].head(10)

Most are ribosomal proteins and other small proteins. On their own they are too small
to have a pocket. Antibiotics that act on the ribosome bind where several proteins and
the ribosomal RNA meet, which a single chain cannot show. Now the distribution of the
best pocket's probability, for the proteins that have one:

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(ranked["top_probability"].dropna(), bins=20, range=(0, 1), color=nc.orange)
ax.axvline(MIN_PROBABILITY, color=nc.gray, linestyle="--")
stylia.label(ax, xlabel="Probability of the best pocket", ylabel="Targets",
             title="How likely is each target's best pocket to bind a ligand?")

The distribution has two peaks: many proteins have a very convincing pocket, and many have only weak ones.

In [ ]:
likely = ranked["top_probability"] >= MIN_PROBABILITY
print(f"{likely.sum()} of {len(ranked)} targets have at least one likely pocket")

### 5.1 Pockets in low-confidence parts of AlphaFold models

A pocket in a part of an AlphaFold model with low pLDDT may not exist in the real
protein: the model may just have left a floppy region in an arbitrary shape.
`pocket_plddt` averages the pLDDT of the residues lining each pocket.

In [ ]:
af = all_pockets["structure_source"] == "AlphaFold"
files = table.set_index("uniprot_ac")["structure_file"]
all_pockets.loc[af, "plddt"] = [pockets.pocket_plddt(files[ac], residues) for ac, residues
                                in all_pockets.loc[af, ["uniprot_ac", "residue_ids"]].values]
low = all_pockets["plddt"] < 70
print(f"{low.sum()} of {af.sum()} AlphaFold pockets are in low-confidence regions (pLDDT < 70)")

How many of those are the best pocket of their protein? Those are the ones that could mislead the ranking.

In [ ]:
top_plddt = all_pockets[all_pockets["rank"] == 1].set_index("uniprot_ac")["plddt"]
ranked["top_plddt"] = ranked["uniprot_ac"].map(top_plddt)
doubtful = ranked[(ranked["top_plddt"] < 70) & likely]
print(f"{len(doubtful)} targets have a likely best pocket in a low-confidence region")
doubtful[["gene_name", "protein_name", "top_probability", "top_plddt"]]

> **Note:** P2Rank's alphafold profile already takes some account of this, but a low
> pLDDT pocket is still best checked by eye, for example by opening the model on the
> [AlphaFold database](https://alphafold.ebi.ac.uk) page of the protein.

## 6. Check against known drug targets

The shortlist includes 12 proteins that existing TB drugs already act on. We know those
proteins *can* be drugged, so a useful pocket score should give most of them a likely
pocket. We never used this information to run P2Rank, so it is a fair check.

In [ ]:
known = ranked[ranked["antibacterial"].notna()]
print(f"{(known['top_probability'] >= MIN_PROBABILITY).sum()} of {len(known)} "
      f"known drug targets have a likely pocket")
known[["gene_name", "antibacterial", "structure_source", "n_likely_pockets",
       "top_probability"]].sort_values("top_probability", ascending=False)

All twelve have a likely pocket, compared with 59% of the whole shortlist, so the
score does favour proteins we know can be drugged. The weakest is `mmpL3`, a membrane
transporter whose inhibitors bind deep inside the part of the protein embedded in the
membrane.

Keep in mind what this check does *not* show. Twelve is a small number, and a likely
pocket somewhere on the protein is not proof that it is the pocket the drug uses.

> **Exercise:** does P2Rank's top pocket match where the drug actually binds? Pick one
> known target, look up a PDB structure of it with its drug bound, and compare the
> binding residues with `top_residues`.

## 7. Rank and save

We rank the targets by the probability of their best pocket, and break ties with the
vulnerability index from the first notebook (more negative is more vulnerable). The
result lists essential, selective targets that have a convincing pocket first.

In [ ]:
ranked = ranked.sort_values(["top_probability", "vi"], ascending=[False, True])
COLUMNS = ["locus_tag", "uniprot_ac", "gene_name", "protein_name", "vi", "antibacterial",
           "structure_source", "structure_id", "chain", "method", "resolution",
           "af_plddt", "p2rank_profile", "n_pockets", "n_likely_pockets", "top_score",
           "top_probability", "top_plddt", "top_residues"]
ranked = ranked[COLUMNS]
ranked[["gene_name", "protein_name", "vi", "top_probability"]].head(10)

We save two files: the ranked targets, and the full table of every pocket. The second
one matters for allosteric sites, which are usually *not* the top pocket. In Colab
both are downloaded to your computer. Upload them to the group's Drive folder
**Projects/OrangeTeam/Data**.

> **Note:** If nothing downloads, your browser may have blocked it. Look for a
> message near the address bar, allow downloads from Colab, and run the
> cell again.

In [ ]:
os.makedirs("outputs", exist_ok=True)
paths = {"outputs/mtb_targets_pockets.csv": ranked, "outputs/mtb_pockets.csv": all_pockets}
for path, df in paths.items():
    df.to_csv(path, index=False)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(path)
    print(f"{len(df):,} rows written to {path}")

## Summary

- P2Rank found the known isoniazid and NADH pocket of InhA as its top pocket, with a high
  probability, without being told anything about the protein.
- Across the 348 targets it predicted 1,772 pockets. 205 targets have at least one likely
  pocket (probability of at least 0.5), and 35, mostly ribosomal and other small
  proteins, have none. For 4 targets the best pocket sits in a low-confidence part of
  an AlphaFold model.
- All 12 targets of existing TB drugs have a likely pocket, against 59% of the
  shortlist overall.
- A pocket score measures whether a small molecule could bind, not whether the protein is
  a good target. It is one piece of evidence, next to essentiality, selectivity and
  novelty.

**Next:** curate the top of the ranked list by hand, and look beyond the top pocket in
`mtb_pockets.csv` for candidate allosteric sites.